In [14]:
import os 
import numpy as np 
import pandas as pd

In [40]:
import warnings
warnings.filterwarnings("ignore")

In [41]:
df = pd.read_csv('C:\\Users\\ander\\Downloads\\goodreads\\books.csv', on_bad_lines='skip')

In [42]:
df.index = df['bookID']

In [43]:
print("Dataset contains {} rows and {} columns".format(df.shape[0], df.shape[1]))

Dataset contains 11123 rows and 12 columns


In [44]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
bookID,,,,,,,,,,,,
1,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
2,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
4,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
5,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
8,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [53]:
import pandas as pd

def popularity_recommender(df, num_recommendations=5, percentile=0.90):

    # Calculate global constants
    C = df['average_rating'].mean()
    m = df['ratings_count'].quantile(percentile)
    
    # Filter out books that don't meet the minimum review threshold
    qualified_books = df[df['ratings_count'] >= m].copy()
    
    # Define the IMDb weighted rating function
    def imdb_weighted_rating(x, m=m, C=C):
        v = x['ratings_count']
        R = x['average_rating']
        return (v / (v + m) * R) + (m / (v + m) * C)
    
    # Apply formula and sort the catalog
    qualified_books['weighted_score'] = qualified_books.apply(imdb_weighted_rating, axis=1)
    recommended_books = qualified_books.sort_values('weighted_score', ascending=False)
    
    # Return the relevant columns
    return recommended_books[['title', 'ratings_count', 'average_rating', 'weighted_score']].head(num_recommendations)

if __name__ == "__main__":

    top_books = popularity_recommender(df, num_recommendations=5, percentile=0.90)
    print(top_books.to_string(index=False))

                                                                title  ratings_count  average_rating  weighted_score
            Harry Potter and the Half-Blood Prince (Harry Potter  #6)        2095690            4.57        4.562576
          Harry Potter and the Prisoner of Azkaban (Harry Potter  #3)        2339585            4.56        4.553447
         Harry Potter and the Order of the Phoenix (Harry Potter  #5)        2153167            4.49        4.483682
               Harry Potter Boxed Set  Books 1-5 (Harry Potter  #1-5)          41428            4.78        4.463604
J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings         101233            4.59        4.461126


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommender(title, df, cosine_sim, indices):
    # Get the index of the book that matches the title
    idx = indices[title]

    # Get the pairwise similarity scores of all books with that book
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the books based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 5 most similar books (excluding the book itself)
    sim_scores = sim_scores[1:6]

    # Get the book indices
    book_indices = [i[0] for i in sim_scores]

    # Return the top 5 most similar books
    return df['title'].iloc[book_indices]



# Apply TF-IDF Vectorizer on the author data
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['authors'])

# Compute the Cosine Similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Construct a reverse map of indices and book titles
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

# Call the function to test recommendations
recommendations = content_based_recommender('The Hobbit', df, cosine_sim, indices)
print(recommendations)

bookID
35924      The Lure of the Basilisk (The Lords of Dûs  #1)
32071                                      Sons and Lovers
2255     Way of the Peaceful Warrior: A Book That Chang...
35724                       The Life You Were Born to Live
11062           Roald Dahl: The Storyteller (Famous Lives)
Name: title, dtype: object
